In [23]:
# Use one GPU with this notebook's TRL 0.23.1 stack. This must run before
# PyTorch is imported. Restart the kernel before using Run All.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [24]:
%pip install transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2


Note: you may need to restart the kernel to use updated packages.


In [25]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this 4-bit QLoRA notebook.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        "Expected one visible GPU. Restart the kernel and run all cells from the top "
        "so CUDA_VISIBLE_DEVICES is applied before PyTorch is imported."
    )

print(f"Visible GPUs: {torch.cuda.device_count()} ({torch.cuda.get_device_name(0)})")

Visible GPUs: 1 (Tesla T4)


In [26]:
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.float32
)

repo_id = 'microsoft/Phi-3-mini-4k-instruct'

model = AutoModelForCausalLM.from_pretrained(
   repo_id, device_map="cuda:0", quantization_config=bnb_config
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
print(model.get_memory_footprint()/1e6)


2206.341312


In [28]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_

In [29]:
# Define the LoRA adapters. SFTTrainer will prepare the quantized model
# and inject this configuration exactly once.
config = LoraConfig(
    # the rank of the adapter, the lower the fewer parameters you'll need to train
    r=8,
    lora_alpha=16, # multiplier, usually 2*r
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    # Newer models, such as Phi-3 at time of writing, may require
    # manually setting target modules
    target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'],
)


In [30]:
print(model.get_memory_footprint()/1e6)


2206.341312


In [31]:
# train_p, tot_p = model.get_nb_trainable_parameters()
# print(f'Trainable parameters:      {train_p/1e6:.2f}M')
# print(f'Total parameters:          {tot_p/1e6:.2f}M')
# print(f'% of trainable parameters: {100*train_p/tot_p:.2f}%')


#### 2. Formatting our dataset

In [32]:
dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset


Dataset({
    features: ['sentence', 'translation', 'translation_extra'],
    num_rows: 720
})

In [33]:
dataset[0]

{'sentence': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'translation_extra': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [34]:
# convert dataset to conversational format
# Adapted from trl.extras.dataset_formatting.instructions_formatting_function
# Converts dataset from prompt/completion format (not supported anymore)
# to the conversational format
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}


In [35]:
dataset = dataset.rename_column("sentence", "prompt")
dataset = dataset.rename_column("translation_extra", "completion")
dataset = dataset.map(format_dataset)
dataset = dataset.remove_columns(['prompt', 'completion', 'translation'])
messages = dataset[0]['messages']
messages


[{'role': 'user', 'content': 'The birch canoe slid on the smooth planks.'},
 {'role': 'assistant',
  'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}]

In [36]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
# Phi-3 already uses EOS as its configured padding token. Keeping the IDs
# aligned avoids the PAD-token configuration warning during trainer.train().
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template


"{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'user' %}{{'<|user|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>\n' + message['content'] + '<|end|>\n'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% else %}{{ eos_token }}{% endif %}"

In [37]:
messages

[{'role': 'user', 'content': 'The birch canoe slid on the smooth planks.'},
 {'role': 'assistant',
  'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}]

In [38]:
print(tokenizer.apply_chat_template(messages, tokenize=False))


<|user|>
The birch canoe slid on the smooth planks.<|end|>
<|assistant|>
On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>
<|endoftext|>


#### 3. Fine-Tuning with SFTTrainer

In [39]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing 
    gradient_checkpointing=True,    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    gradient_checkpointing_kwargs={'use_reentrant': False}, 
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=1,  
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=16, 
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=False,

    ## GROUP 2: Dataset-related
    max_length=64, # renamed in v0.20
    # Dataset
    # packing a dataset means no padding is needed
    packing=False,    #True,
    # packing_strategy='wrapped', # added to approximate original packing behavior

    ## GROUP 3: These are typical training parameters
    num_train_epochs=10,
    learning_rate=3e-4,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='paged_adamw_8bit',       
    
    ## GROUP 4: Logging parameters
    logging_steps=10,
    logging_dir='./logs',
    output_dir='./phi3-mini-yoda-adapter',
    report_to='none',

    # ensures bf16 (the new default) is only used when it is actually available
    bf16=torch.cuda.is_bf16_supported(including_emulation=False)
)


In [40]:
trainer = SFTTrainer(
    model=model, # the underlying Phi-3 model
    peft_config=config,  # added to fix issue in TRL>=0.20
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset,
)


In [41]:
# Sanity checks: a notebook process using DataParallel reproduces the
# TRL 0.23.1 entropy mismatch (per-device batch 2 vs global batch 4).
print(f"Per-device batch size: {trainer.args.per_device_train_batch_size}")
print(f"Visible Trainer GPUs: {trainer.args.n_gpu}")
print(f"Effective train batch size: {trainer.args.train_batch_size}")

assert trainer.args.n_gpu == 1, (
    "Restart the kernel and run all cells from the top. The trainer still sees "
    "more than one GPU."
)
trainer.model.print_trainable_parameters()


Per-device batch size: 16
Visible Trainer GPUs: 1
Effective train batch size: 16
trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


In [42]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))


In [43]:
batch['input_ids'][0], batch['labels'][0]


(tensor([32010,  2688,  3614,  1009,   413,  4841,   515,   278,   970,  3762,
         29889, 32007, 32001,  3645,   278,   970,  3762, 29892,  1009,   413,
          4841,   896,  3614, 29889,   379,  1758,  4317, 29889, 32007, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000],
        device='cuda:0'),
 tensor([32010,  2688,  3614,  1009,   413,  4841,   515,   278,   970,  3762,
         29889, 32007, 32001,  3645,   278,   970,  3762, 29892,  1009,   413,
          4841,   896,  3614, 29889,   379,  1758,  4317, 29889, 32007, 32000,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
        device='cuda:0'))

In [44]:
trainer.train()


Step,Training Loss
10,2.360300
20,1.465300
30,1.375700
40,1.302200
50,1.229600
60,1.101500
70,1.132000
80,1.128900
90,1.128600
100,0.821800


TrainOutput(global_step=450, training_loss=0.5669216023551094, metrics={'train_runtime': 2339.8007, 'train_samples_per_second': 3.077, 'train_steps_per_second': 0.192, 'total_flos': 6247461601640448.0, 'train_loss': 0.5669216023551094, 'epoch': 10.0})

#### 4. Inference

In [45]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [{"role": "user", "content": sentence}]
    prompt = tokenizer.apply_chat_template(
        converted_sample, tokenize=False, add_generation_prompt=True
    )
    return prompt


In [46]:
sentence = 'The Force is strong in you!'
prompt = gen_prompt(tokenizer, sentence)
print(prompt)


<|user|>
The Force is strong in you!<|end|>
<|assistant|>



In [49]:
from contextlib import nullcontext

In [52]:
def generate(model, tokenizer, prompt, max_new_tokens=64, skip_special_tokens=False):
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    # if it was trained using mixed precision, uses autocast context
    ctx = torch.autocast(device_type=model.device.type, dtype=model.dtype) \
          if model.dtype in [torch.float16, torch.bfloat16] else nullcontext()
    with ctx:  
        gen_output = model.generate(**tokenized_input,
                                    eos_token_id=tokenizer.eos_token_id,
                                    max_new_tokens=max_new_tokens)
    
    output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)
    return output[0]


In [53]:
print(generate(model, tokenizer, prompt))


<|user|> The Force is strong in you!<|end|><|assistant|> Strong in you, the Force is!<|end|><|endoftext|>
